In [1]:
# ============================================================
# AGENTIC AI BANKING CUSTOMER GROWTH PLATFORM
# Integrated 5-Agent Pipeline
# ============================================================

import os
from dotenv import load_dotenv
import anthropic
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()
client = anthropic.Anthropic()

print("✅ Pipeline initialised")

✅ Pipeline initialised


In [2]:
# Load all bank products into vector database
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

db = chromadb.Client()
products = db.create_collection("bank_products", embedding_function=embedding_fn)

products.add(
    ids=["PROD_001", "PROD_002", "PROD_003", "PROD_004", "PROD_005"],
    documents=[
        "Premium travel credit card with airport lounge access, zero foreign transaction fees, travel insurance, and rewards on international spending",
        "Multi-currency foreign exchange account for frequent international transfers, competitive exchange rates, no transfer fees",
        "High-interest savings account for customers wanting to build financial discipline and grow emergency funds",
        "Flexible personal loan for large purchases or debt consolidation with competitive rates",
        "Family current account with joint spending features, children's pocket money tools, and family budgeting"
    ],
    metadatas=[
        {"name": "Voyager Premium Travel Card", "id": "PROD_001"},
        {"name": "GlobalCash FX Account", "id": "PROD_002"},
        {"name": "HomeSaver Account", "id": "PROD_003"},
        {"name": "FlexiLoan Personal Loan", "id": "PROD_004"},
        {"name": "FamilyFirst Current Account", "id": "PROD_005"},
    ]
)

print("✅ Product database ready — 5 products loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Product database ready — 5 products loaded


In [3]:
def run_pipeline(customer_data: str) -> dict:
    """
    Full agentic pipeline. One call triggers all 5 agents in sequence.
    Input:  raw customer data string
    Output: complete Opportunity Card ready for executive
    """

    print("\n" + "="*60)
    print("🚀 PIPELINE STARTING")
    print("="*60)

    # ── AGENT 1: Customer Intelligence ──────────────────────────
    print("\n🔍 Agent 1: Analysing customer...")
    
    a1_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": f"""Analyse this customer's transaction data and produce a structured profile.

Customer Data:
{customer_data}

Return ONLY this format:
CUSTOMER_ID: [id]
NAME: [name]
PRIMARY_NEED: [one sentence describing their main financial need]
SEARCH_QUERY: [a descriptive phrase for finding relevant products, e.g. "frequent international traveller with FX needs"]
AFFINITIES: Travel:[HIGH/MED/LOW] | FX:[HIGH/MED/LOW] | Savings:[HIGH/MED/LOW] | Premium:[HIGH/MED/LOW]
PRIVACY_CHECK: PASSED"""
        }]
    )
    
    agent1_output = a1_response.content[0].text
    print(agent1_output)

    # Extract search query from Agent 1 output
    search_query = ""
    for line in agent1_output.split("\n"):
        if line.startswith("SEARCH_QUERY:"):
            search_query = line.replace("SEARCH_QUERY:", "").strip()
            break

    # ── AGENT 2: Product Intelligence ───────────────────────────
    print("\n🗄️  Agent 2: Searching product database...")
    
    results = products.query(query_texts=[search_query], n_results=2)
    matched_products = []
    for i, doc in enumerate(results["documents"][0]):
        name = results["metadatas"][0][i]["name"]
        matched_products.append(f"- {name}: {doc}")
        print(f"  Match: {name}")

    # ── AGENT 3: Opportunity Discovery ──────────────────────────
    print("\n💡 Agent 3: Discovering opportunity...")
    
    a3_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""Given this customer profile and matched products, identify the single best opportunity.

Customer Profile:
{agent1_output}

Matched Products:
{chr(10).join(matched_products)}

Return ONLY:
BEST_PRODUCT: [product name]
FIT_REASON: [one sentence]
CONVERSATION_READY: YES/NO
REASON: [one sentence]"""
        }]
    )
    
    agent3_output = a3_response.content[0].text
    print(agent3_output)

    # ── AGENT 4: Offer Construction ──────────────────────────────
    print("\n🎯 Agent 4: Constructing personalised offer...")
    
    a4_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Construct a personalised offer for the executive to present.

Customer Profile:
{agent1_output}

Opportunity:
{agent3_output}

Return ONLY:
VALUE_PROPOSITION: [what the customer gains, in their terms]
KEY_BENEFITS: [3 bullet points]
OBJECTION_HANDLER: [one likely objection and how to address it]"""
        }]
    )
    
    agent4_output = a4_response.content[0].text
    print(agent4_output)

    # ── AGENT 5: Conversation Coach ──────────────────────────────
    print("\n🗣️  Agent 5: Coaching executive...")
    
    a5_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""Write the opening line an executive should say to this customer right now.

Customer Profile:
{agent1_output}

Offer:
{agent4_output}

Rules:
- Must feel personal, not scripted
- Reference a specific behaviour pattern (never a specific transaction)
- Under 60 words
- End with a question

Return ONLY the opening line, nothing else."""
        }]
    )
    
    opening_line = a5_response.content[0].text.strip()

    # ── FINAL OUTPUT: Opportunity Card ───────────────────────────
    print("\n" + "="*60)
    print("📋 OPPORTUNITY CARD — READY FOR EXECUTIVE")
    print("="*60)
    
    # Extract best product name
    best_product = ""
    for line in agent3_output.split("\n"):
        if line.startswith("BEST_PRODUCT:"):
            best_product = line.replace("BEST_PRODUCT:", "").strip()
            break

    card = {
        "customer_profile": agent1_output,
        "recommended_product": best_product,
        "opportunity": agent3_output,
        "offer": agent4_output,
        "opening_line": opening_line
    }

    print(f"\n✅ RECOMMENDED PRODUCT: {best_product}")
    print(f"\n💬 OPENING LINE:\n{opening_line}")
    print("\n" + "="*60)
    
    return card

In [4]:
# Sarah Mitchell's customer data
sarah = """
Customer ID: CUST_001
Name: Sarah Mitchell
Age: 34
Current Products: Basic Debit Card

Transaction Pattern (last 6 months):
- 4 international flights booked (Europe, Asia, Americas)
- Regular FX transactions in USD, EUR, JPY
- Airport lounge attempts (access denied — no lounge card)
- Monthly international hotel bookings
- Frequent foreign restaurant and retail spending
- No savings products held
- Income: £4,200/month, stable
"""

result = run_pipeline(sarah)


🚀 PIPELINE STARTING

🔍 Agent 1: Analysing customer...
CUSTOMER_ID: CUST_001
NAME: Sarah Mitchell
PRIMARY_NEED: Access to premium travel benefits and optimized foreign exchange services to support frequent international business or leisure travel.
SEARCH_QUERY: Frequent international traveller with high FX transaction volume seeking travel rewards and lounge access
AFFINITIES: Travel:[HIGH] | FX:[HIGH] | Savings:[LOW] | Premium:[HIGH]
PRIVACY_CHECK: PASSED

🗄️  Agent 2: Searching product database...
  Match: Voyager Premium Travel Card
  Match: GlobalCash FX Account

💡 Agent 3: Discovering opportunity...
BEST_PRODUCT: Voyager Premium Travel Card
FIT_REASON: Directly addresses both primary needs (travel benefits and FX optimization) while aligning with HIGH affinity for travel and premium offerings, plus includes lounge access and travel insurance valued by frequent international travelers.
CONVERSATION_READY: YES
REASON: Customer profile clearly indicates strong demand signals for prem

In [5]:
# Test with a completely different customer — savings needs, not travel
james = """
Customer ID: CUST_002
Name: James Okafor
Age: 28
Current Products: Basic Current Account

Transaction Pattern (last 6 months):
- Frequent small overdrafts (6 times in 6 months)
- Payday loan payment appearing monthly
- No savings account held
- Spending spikes in first week of month, drops sharply after
- Regular takeaway and subscription spending
- Income: £2,100/month, stable
- No international transactions
- No travel spending
"""

result2 = run_pipeline(james)


🚀 PIPELINE STARTING

🔍 Agent 1: Analysing customer...
CUSTOMER_ID: CUST_002
NAME: James Okafor
PRIMARY_NEED: Help managing cash flow and reducing reliance on payday loans through better budgeting tools and overdraft alternatives.
SEARCH_QUERY: Frequent overdraft user with irregular spending patterns seeking cash flow management solutions
AFFINITIES: Travel:LOW | FX:LOW | Savings:HIGH | Premium:LOW
PRIVACY_CHECK: PASSED

🗄️  Agent 2: Searching product database...
  Match: HomeSaver Account
  Match: FamilyFirst Current Account

💡 Agent 3: Discovering opportunity...
BEST_PRODUCT: HomeSaver Account

FIT_REASON: Directly addresses the primary need for better cash flow management by helping build emergency funds to reduce overdraft dependency, while aligning with the customer's high savings affinity.

CONVERSATION_READY: YES

REASON: Clear alignment between customer's stated pain point (overdraft reliance), product capability (emergency fund building), and demonstrated behavior (irregular s